# Experiment 2: 50 baseline simulation runs

Run **Runtime → Run all** in Colab. This self-contained notebook repeats the same model over 50 seeds. No animation or local installation is needed. It reuses the baseline rules; it does not yet train a predictor.

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.colors import Normalize
from IPython.display import HTML, display

SEED = 42
N = 100
MEAN_DEGREE = 6
CAPACITY = 10.0
INITIAL_UTILIZATION = (0.4, 0.7)
DEMAND_INCREMENT = 1.0
COLLAPSE_FRACTION = 0.30
MAX_STEPS = 1000

In [ ]:
def resolve_cascade(graph, loads, capacities, active, unserved=0.0):
    """Return a settled copy and a trace of synchronous failure rounds."""
    loads = dict(loads)
    active = set(active)
    trace = []
    while True:
        overloaded = {i for i in active if loads[i] > capacities[i]}
        if not overloaded:
            break
        survivors = active - overloaded
        additions = {i: 0.0 for i in survivors}
        for i in sorted(overloaded):
            recipients = sorted(set(graph.neighbors(i)) & survivors)
            if recipients:
                share = loads[i] / len(recipients)
                for j in recipients:
                    additions[j] += share
            else:
                unserved += loads[i]
            loads[i] = 0.0
        for j, amount in additions.items():
            loads[j] += amount
        active = survivors
        trace.append({'active': set(active), 'loads': dict(loads),
                      'failed': sorted(overloaded), 'unserved': unserved})
    return loads, active, unserved, trace

In [ ]:
def simulate(seed=SEED, n=N, mean_degree=MEAN_DEGREE,
             capacity=CAPACITY, utilization=INITIAL_UTILIZATION,
             delta_d=DEMAND_INCREMENT, collapse_fraction=COLLAPSE_FRACTION,
             max_steps=MAX_STEPS):
    if not (n >= 2 and 0 < mean_degree <= n-1 and capacity > 0
            and 0 <= utilization[0] <= utilization[1] < 1
            and delta_d > 0 and 0 < collapse_fraction <= 1 and max_steps >= 1):
        raise ValueError('Invalid simulation settings')
    rng = np.random.default_rng(seed)
    for attempt in range(10000):
        graph = nx.gnp_random_graph(n, mean_degree/(n-1),
                                   seed=int(rng.integers(0, 2**32-1)))
        if nx.is_connected(graph):
            break
    else:
        raise RuntimeError('Could not sample connected graph; revise connectivity')
    capacities = dict.fromkeys(graph, float(capacity))
    loads = {i: capacity * rng.uniform(*utilization) for i in graph}
    active = set(graph)
    initial = sum(loads.values())
    unserved = injected = 0.0
    history, frames = [], []
    collapse_step = None

    def frame(step, phase, failed=()):
        frames.append(dict(step=step, phase=phase, active=set(active),
                           loads=dict(loads), failed=list(failed)))

    def record(step, cascade_size):
        active_total = sum(loads[i] for i in active)
        assert np.isclose(active_total + unserved, initial + injected)
        spare = sum(capacities[i] - loads[i] for i in active)
        near = sum(loads[i]/capacities[i] >= .9 for i in active)
        giant = max((len(c) for c in nx.connected_components(graph.subgraph(active))), default=0)
        history.append(dict(step=step, injected=injected, failed=n-len(active),
                            failed_fraction=1-len(active)/n, spare=spare/(n*capacity),
                            near_fraction=near/len(active) if active else np.nan,
                            unserved=unserved, active_load=active_total,
                            cascade_size=cascade_size, giant_fraction=giant/n))

    record(0, 0)
    frame(0, 'initial')
    for step in range(1, max_steps+1):
        before = len(active)
        for i in active:
            loads[i] += delta_d / before
        injected += delta_d
        frame(step, 'demand added')
        loads, active, unserved, trace = resolve_cascade(
            graph, loads, capacities, active, unserved)
        for round_number, state in enumerate(trace, 1):
            assert np.isclose(sum(state['loads'][i] for i in state['active'])
                              + state['unserved'], initial + injected)
            frames.append(dict(step=step, phase=f'cascade round {round_number}',
                               active=state['active'], loads=state['loads'],
                               failed=state['failed']))
        record(step, before-len(active))
        if (n-len(active)) >= collapse_fraction*n - 1e-9:
            collapse_step = step
            break
    return dict(graph=graph, capacities=capacities, history=history,
                frames=frames, collapse_step=collapse_step,
                initial_load=initial, attempts=attempt+1)

result = simulate()
repeat = simulate()
assert result['graph'].edges == repeat['graph'].edges
assert result['history'] == repeat['history']
G = result['graph']
print(f'Nodes: {len(G)}, edges: {G.number_of_edges()}, mean degree: {2*G.number_of_edges()/len(G):.2f}')
print(f'Connected graph sampling attempts: {result["attempts"]}')
print(f'Collapse step: {result["collapse_step"]}; final failed nodes: {result["history"][-1]["failed"]}')
print('Workload accounting and seeded reproducibility checks passed.')

## Experiment 2 — Repeat the baseline over 50 random seeds

**Question:** Is abrupt collapse typical under these baseline assumptions, and how much does collapse timing vary?

Keep node count, connectivity target, capacity, initial-load range, demand increment, and collapse definition fixed. Change only the random seed. Each seed regenerates **both the network and initial loads**, so this measures their combined variability; it does not isolate topology from load variation.

Run the baseline definition cells first, then the following two cells. No extra packages are needed. We save summary measurements and settled-state histories as CSV files in the Colab runtime. These are pilot runs, not a training/test evaluation of a predictor.

The plots aligned by actual remaining time use the eventual collapse step **only for retrospective analysis**. A future predictor must not receive that information as an input. Runs without observed collapse are retained and excluded only from plots requiring a known collapse time.

In [ ]:
import csv
from pathlib import Path

EXPERIMENT_SEEDS = range(1000, 1050)
experiment_summaries = []
experiment_histories = {}

for index, seed in enumerate(EXPERIMENT_SEEDS, 1):
    run = simulate(seed=seed)
    history = run['history']
    collapse_step = run['collapse_step']
    first_failure = next((r['step'] for r in history if r['failed'] > 0), None)
    collapsed_same_step = (first_failure == collapse_step) if collapse_step is not None else None
    prior = history[-2] if collapse_step is not None and len(history) >= 2 else None
    experiment_summaries.append({
        'seed': seed,
        'collapse_observed': collapse_step is not None,
        'collapse_step': collapse_step,
        'last_observed_step': history[-1]['step'],
        'first_failure_step': first_failure,
        'steps_from_first_failure_to_collapse': collapse_step-first_failure
            if collapse_step is not None and first_failure is not None else None,
        'collapse_in_first_failure_step': collapsed_same_step,
        'largest_cascade_nodes': max(r['cascade_size'] for r in history),
        'failed_nodes_at_end': history[-1]['failed'],
        'near_overload_percent_one_step_before_collapse':
            100*prior['near_fraction'] if prior is not None else None,
        'actual_mean_degree': 2*run['graph'].number_of_edges()/len(run['graph']),
    })
    # Retain compact histories; do not retain all animation frames across 50 runs.
    experiment_histories[seed] = history
    if index % 10 == 0:
        print(f'Completed {index}/50 runs')

del run
collapsed = [r for r in experiment_summaries if r['collapse_observed']]
collapse_times = np.array([r['collapse_step'] for r in collapsed], dtype=float)
print(f'Observed collapse: {len(collapsed)}/50 runs')
if len(collapse_times):
    print(f'Collapse step: median {np.median(collapse_times):.1f}; '
          f'range {collapse_times.min():.0f}–{collapse_times.max():.0f}')
    print(f'Mean {collapse_times.mean():.1f}; '
          f'sample standard deviation {collapse_times.std(ddof=1):.1f}'
          if len(collapse_times)>1 else 'Only one observed collapse')
    immediate = sum(r['collapse_in_first_failure_step'] for r in collapsed)
    print(f'Collapse in the same external step as first failure: {immediate}/{len(collapsed)} collapse runs')

output_dir = Path('experiment_results')
output_dir.mkdir(exist_ok=True)
with (output_dir/'baseline_50_run_summary.csv').open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(experiment_summaries[0]))
    writer.writeheader()
    writer.writerows(experiment_summaries)
with (output_dir/'baseline_50_run_histories.csv').open('w', newline='') as f:
    keys = ['seed', 'collapse_step', 'steps_remaining'] + list(next(iter(experiment_histories.values()))[0])
    writer = csv.DictWriter(f, fieldnames=keys)
    writer.writeheader()
    for row in experiment_summaries:
        for state in experiment_histories[row['seed']]:
            writer.writerow({'seed': row['seed'], 'collapse_step': row['collapse_step'],
                             'steps_remaining': row['collapse_step']-state['step']
                                 if row['collapse_observed'] else None, **state})
print('CSV files saved in experiment_results/ (Colab Files panel).')

In [ ]:
with plt.style.context('seaborn-v0_8-whitegrid'):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    if len(collapse_times):
        axes[0].hist(collapse_times, bins='auto', color='#2563eb', edgecolor='white')
        axes[0].axvline(np.median(collapse_times), color='#be123c', linestyle='--', label='Median')
        axes[0].legend()
    else:
        axes[0].text(.5, .5, 'No collapse observed', ha='center', transform=axes[0].transAxes)
    axes[0].set_title('When did collapse happen?', loc='left', fontweight='bold')
    axes[0].set_xlabel('Collapse step')
    axes[0].set_ylabel('Number of runs')

    # A fixed selection of the first 10 seeds, not chosen for visual drama.
    for row in experiment_summaries[:10]:
        history = experiment_histories[row['seed']]
        axes[1].step([r['step'] for r in history], [r['failed'] for r in history],
                     where='post', alpha=.65, label=f"Seed {row['seed']}")
    axes[1].axhline(COLLAPSE_FRACTION*N, color='#64748b', linestyle=':', label='Collapse threshold')
    axes[1].set_title('Gradual damage or sudden failure?', loc='left', fontweight='bold')
    axes[1].set_xlabel('Simulation step')
    axes[1].set_ylabel(f'Total failed nodes (out of {N})')
    axes[1].set_ylim(0, N*1.05)

    # Use only PRE-collapse observations from the final 100 steps.
    aligned = {}
    for row in collapsed:
        for state in experiment_histories[row['seed']]:
            remaining = row['collapse_step']-state['step']
            if 1 <= remaining <= 100 and np.isfinite(state['near_fraction']):
                aligned.setdefault(remaining, []).append(100*state['near_fraction'])
    if aligned:
        remaining = np.array(sorted(aligned))
        median = [np.median(aligned[k]) for k in remaining]
        low = [np.percentile(aligned[k], 10) for k in remaining]
        high = [np.percentile(aligned[k], 90) for k in remaining]
        axes[2].fill_between(remaining, low, high, color='#d97706', alpha=.2,
                             label='10th–90th percentile across available runs')
        axes[2].plot(remaining, median, color='#d97706', linewidth=2.5, label='Median')
        axes[2].invert_xaxis()
        axes[2].legend(fontsize=8)
    axes[2].set_title('Stress before collapse — retrospective', loc='left', fontweight='bold')
    axes[2].set_xlabel('Actual steps remaining until collapse →')
    axes[2].set_ylabel('Active nodes using ≥90% capacity (%)')
    axes[2].set_ylim(0, 100)
    for ax in axes:
        ax.spines[['top', 'right']].set_visible(False)
    fig.suptitle('50 independent seeded runs — same baseline settings', fontsize=16, fontweight='bold')
    fig.tight_layout(rect=(0, 0, 1, .94))
    plt.show()

print('Read left to right: variation in lifetime; failure trajectories; stress as collapse approaches.')
print('The shaded band shows variation between runs, not a calibrated prediction interval.')
print('If collapse nearly always follows the first failure in the same step, test capacity heterogeneity next.')